# Two entities form a specialization cycle

Can two individually consistent relationships form a cycle?

Run cells from top to bottom in a fresh Python kernel using the repository environment. Set the working directory to the repository root. This notebook is self-contained; do not run another declaration experiment in the same kernel. The matching `.py` file runs with `uv run python examples/step_21_relations/37_error_cycle.py`.

This is a deliberately broken declaration. The error is caught and printed; the corrected declaration is explained at the end. Read the [diagnostic guide](../../docs/how-to/specialization-declaration-fails.md) for the working baseline.


## Imports

These imports supply the annotation tools and AOA types used below.

In [1]:
from __future__ import annotations

from typing import Annotated, Literal

from pydantic import Field

from aoa.action_machine.domain import BaseEntity, Classifier, Generalization, Inverse, Rel, Specialization
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity


## MusicDomain

Group the cycle experiment. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [2]:
class MusicDomain(BaseDomain):
    """Group the cycle experiment."""

    name = "music"
    description = "Cycle experiment"


## AEntity

Declare the first level of detail. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [3]:
@entity(description="First level", domain=MusicDomain)
class AEntity(BaseEntity):
    """Declare the first level of detail."""

    id: str = Field(description="Identifier")
    kind: str = Field(description="Detail code")
    detail: Annotated[Specialization[BEntity], Classifier("kind", Literal["b"]), Inverse(field_name="back")] = Rel(
        description="Next level"
    )
    back: Annotated[Generalization[BEntity], Classifier("back", Literal["a"]), Inverse(BEntity, "detail")] = Rel(
        description="Return to B"
    )


## BEntity

Declare the second level of detail. The field descriptions explain the data each object carries. References to classes defined later are resolved after all declarations are present.

In [4]:
@entity(description="Second level", domain=MusicDomain)
class BEntity(BaseEntity):
    """Declare the second level of detail."""

    id: str = Field(description="Identifier")
    kind: str = Field(description="Detail code")
    back: Annotated[Generalization[AEntity], Classifier("back", Literal["b"]), Inverse(AEntity, "detail")] = Rel(
        description="Return to A"
    )
    detail: Annotated[Specialization[AEntity], Classifier("kind", Literal["a"]), Inverse(field_name="back")] = Rel(
        description="Next level"
    )


## Resolve the declared types

Pydantic can now resolve the references between classes. This does not create record objects or load data.

In [5]:
AEntity.model_rebuild()


## Resolve the declared types

Pydantic can now resolve the references between classes. This does not create record objects or load data.

In [6]:
BEntity.model_rebuild()


## The experiment

Can two individually consistent relationships form a cycle?

In [7]:
from aoa.action_machine.domain.exceptions import SpecializationDeclarationError
from aoa.action_machine.graph.validators.entity_specialization_validator import validate_entity_specializations

try:
    validate_entity_specializations()
except SpecializationDeclarationError as error:
    print(type(error).__name__ + ": " + str(error))
else:
    raise AssertionError("The broken declaration was accepted")


SpecializationDeclarationError: Specialization 'detail' on entity 'AEntity' is invalid: the head and its alternatives form a cycle: AEntity -> BEntity -> AEntity


## Why this result occurs

The codes agree on both relationships, but following detail leads from A to B and back to A. This real two-class model reaches the cycle check.

## Repair

Remove the B-to-A specialization: remove BEntity.detail and its reverse AEntity.back. A-to-B remains valid.

Apply that change to the declaration above, restart the kernel, and run again. Replace the experiment cell with the following success check. This corrected model has also been run through an actual machine build:

```python
from aoa.action_machine.graph.validators.entity_specialization_validator import validate_entity_specializations
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

validate_entity_specializations()
ActionProductMachine(loggers=[])
print("Corrected model built")
```

Expected output:

```text
Corrected model built
```